# Takealot dataset (standalone)

This notebook **does not** import `src.utils` or `run_all.py`.

**Paths:** set `DATA_ROOT` in the next cell if the loader cannot find your CSVs. Otherwise it uses `TAKEALOT_DATA_DIR`, then `data/take-a-lot-dataset` or `data/takealot` (cwd or project root), then `~/Documents/take-a-lot-dataset`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Resolve standalone_takealot on the path (notebook cwd may be repo root or this folder)
_cwd = Path.cwd()
for _p in (_cwd, _cwd / "standalone_takealot"):
    if (_p / "takealot_loader.py").is_file():
        sys.path.insert(0, str(_p.resolve()))
        break
else:
    raise FileNotFoundError("Could not find takealot_loader.py — open the Thesis_Research folder or standalone_takealot as cwd")

from takealot_loader import (
    load_takealot,
    load_reviews_raw,
    ratings_from_reviews_df,
    resolve_takealot_root,
    split_summary,
)

# Set to a string path if auto-detection fails, e.g. r"C:\Users\you\Documents\take-a-lot-dataset"
DATA_ROOT = None

root = resolve_takealot_root(DATA_ROOT)
print("Takealot root:", root)

In [ ]:
summary = split_summary(root)
summary

In [ ]:
train_ratings, item_feats = load_takealot("train", root, dedupe_user_item=True)
test_ratings, _ = load_takealot("test", root, dedupe_user_item=True)

print(train_ratings.shape, test_ratings.shape)
print("Train users:", train_ratings["user_id"].nunique(), "items:", train_ratings["item_id"].nunique())
train_ratings.head()

In [ ]:
# Item popularity on TRAIN (typical input to a recommender)
pop = train_ratings.groupby("item_id").size().rename("n").sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(pop.values, bins=50, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].set_yscale("log")
axes[0].set_xlabel("Interactions per item")
axes[0].set_ylabel("Item count (log)")
axes[0].set_title("Train popularity histogram")

axes[1].loglog(np.arange(1, len(pop) + 1), pop.values, color="darkred", lw=1.2)
axes[1].set_xlabel("Rank")
axes[1].set_ylabel("Interactions")
axes[1].set_title("Train popularity (log-log)")
plt.tight_layout()
plt.show()

top_share = pop.sort_values(ascending=False).iloc[: max(1, int(0.1 * len(pop)))].sum() / pop.sum()
print(f"Top 10% of items hold {100*top_share:.1f}% of train interactions")

In [ ]:
# Rating distribution (train)
train_ratings["rating"].value_counts().sort_index().plot(kind="bar", color="seagreen", figsize=(6, 3))
plt.xlabel("Stars")
plt.ylabel("Count")
plt.title("Train review_rating distribution")
plt.tight_layout()
plt.show()

In [ ]:
# Sample item metadata (from products.csv)
item_feats.head(10)